# Zu einem lokalen Modell wechseln

In diesem Notebook werden wir:

1. Ein lokales KI-Modell mit Ollama starten.
2. Dieselben 10 Rezensionen sowohl mit dem lokalen Modell als auch mit dem API-Modell klassifizieren.
3. Die beiden Modelle hinsichtlich Genauigkeit, Geschwindigkeit, Kosten und Datenschutz vergleichen.

**Daten:** Wir verwenden die Datei `data/processed/silver_labels.csv`, die bereits im Notebook [classify_reviews.ipynb](../03_Classify_Reviews/classify_reviews.ipynb) erstellt wurde. Wir verwenden diese Datei erneut. Eine zusätzliche Datenaufbereitung ist nicht notwendig.

## Bibliotheken und Einstellungen

In [1]:
# Libraries
import os
import subprocess
import time
import platform
from datetime import datetime
from pathlib import Path
from platform import python_version

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import accuracy_score

# Folder with the data (relative to the folder of this notebook)
DATA = Path("../data")

# API model and course price assumptions in USD per 1 million tokens
MODEL = "gpt-6-luna"
PRICE_INPUT_PER_M = 0.10
PRICE_OUTPUT_PER_M = 0.50

# Local model: small enough for a codespace without a graphics card
LOCAL_MODEL = "qwen3:1.7b"

# The API key is a Codespaces secret; outside Codespaces it is read from .env
load_dotenv("../.env")

# The client sends our requests and retries when the rate limit is reached
client = OpenAI(max_retries=8)

# Show current working directory
print(os.getcwd())

/workspaces/llm_fundamentals/05_Local_Model


## Schritt 1: Ollama starten und das Modell herunterladen

Die Codezelle führt drei Schritte aus:

1. Ollama installieren, falls es noch nicht vorhanden ist. Im Codespace ist Ollama bereits installiert.
2. Den Ollama-Server starten, damit das lokale Modell verwendet werden kann.
3. Das Modell herunterladen. Es ist ungefähr 1,4 GB gross und muss nur beim ersten Mal heruntergeladen werden.

Zeilen, die mit `!` beginnen, sind Shell-Befehle. Damit können wir direkt aus einem Jupyter-Notebook Befehle im Betriebssystem ausführen.

In [2]:
# 1. Install Ollama if it is missing (about 2 minutes)
!which ollama > /dev/null || curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server in the background.
# If the server is already running, this does nothing.
subprocess.Popen(
    ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

# 3. Download the model (only the first time) and list the installed models
!ollama pull {LOCAL_MODEL} 2>&1 | tail -n 1
!ollama list

success 
]11;?\NAME          ID              SIZE      MODIFIED       
qwen3:1.7b    8f68893c685c    1.4 GB    18 seconds ago    


### Den Client verbinden

Ollama ist auf diesem Computer über die Adresse `localhost:11434` erreichbar.

Wir verwenden dieselbe OpenAI-Bibliothek wie zuvor, geben aber eine lokale Adresse (`base_url`) und den Namen des lokalen Modells an.

Ollama ignoriert den API-Schlüssel. Die OpenAI-Bibliothek verlangt jedoch, dass einer angegeben wird.

In [3]:
# A client that talks to Ollama on this computer
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


## Schritt 2: 10 Rezensionen mit beiden Modellen klassifizieren

Wir verwenden 10 Rezensionen mit Silver Labels aus Notebook 03:

- 5 Rezensionen von IKEA
- 5 Rezensionen von McDonald’s

Beide Modelle erhalten denselben Prompt, damit wir ihre Ergebnisse vergleichen können.

In [4]:
# Reviews with silver labels from notebook 03
silver_path = DATA / "processed/silver_labels.csv"
silver = pd.read_csv(silver_path)

# 5 random reviews per chain, the same inputs for both models
test = silver.groupby("chain").sample(n=5, random_state=42)
test = test.reset_index(drop=True)

# The same classification prompt for both models
PROMPT = (
    "Classify the sentiment of this customer review. "
    "Answer with exactly one word: positive, negative or mixed.\n"
    "\n"
    "Review: "
)

test[["id", "chain", "stars", "silver", "text"]].head(5)

,id,chain,stars,silver,text
0,R009,IKEA,3.0,negative,I was excited about making a good review. Howe...
1,R185,IKEA,1.0,negative,I want to start by saying before this experien...
2,R152,IKEA,1.0,mixed,"Wow, the customer service here has fallen apar..."
3,R028,IKEA,3.0,positive,Good furniture. I like how cheap everything is...
4,R079,IKEA,1.0,negative,Well don't even try to order a new mattress fr...


### Lokales Modell

Bevor wir die Antworten des lokalen Modells mit den Silver Labels vergleichen, bereinigen wir sie.

Dabei werden:
- Leerzeichen am Anfang und Ende entfernt.
- Alle Buchstaben in Kleinbuchstaben umgewandelt.
- Ein Punkt am Ende der Antwort entfernt.

Wenn die bereinigte Antwort danach nicht exakt einem der drei Labels (positive, negative oder mixed) entspricht, wird sie als Fehler gezählt.

In [5]:
# Local model: one review after the other, and we stop the time
labels = []
start = time.perf_counter()

for text in test["text"]:
    r = local.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[{"role": "user", "content": PROMPT + text}],
        reasoning_effort="none",
    )

    # Clean the answer: no spaces around it, lower case, no final period
    answer = r.choices[0].message.content
    answer = answer.strip().lower()
    labels.append(answer.rstrip("."))

local_seconds = (time.perf_counter() - start) / len(test)
test["local"] = labels

print("Seconds per review:", round(local_seconds, 2))

Seconds per review: 5.32


### API-Modell

Dieselben Rezensionen werden anschliessend an das API-Modell gesendet.

Mit `store=False` wird der API-Anbieter angewiesen, die Anfrage nicht als gespeicherte Antwort für den späteren Abruf aufzubewahren.

Wichtig: **Bei jeder erneuten Ausführung dieser Codezelle werden neue API-Anfragen gesendet, wodurch zusätzliche Kosten entstehen.**

Die Höhe dieser Kosten hängt von der Länge der Rezensionstexte und der Anzahl der generierten Output-Tokens ab.

Die nachfolgende Berechnung schätzt die Kosten anhand der im Kurs verwendeten Preise.

In [6]:
# API model: the same reviews, and we add up the tokens
labels = []
input_tokens = 0
output_tokens = 0
start = time.perf_counter()

for text in test["text"]:
    r = client.responses.create(
        model=MODEL,
        input=PROMPT + text,
        reasoning={"effort": "low"},
        store=False,
    )

    # Clean the answer in the same way as for the local model
    answer = r.output_text.strip().lower()
    labels.append(answer.rstrip("."))

    input_tokens += r.usage.input_tokens
    output_tokens += r.usage.output_tokens

api_seconds = (time.perf_counter() - start) / len(test)
test["api"] = labels

# Cost: tokens / 1 million x price per 1 million tokens
api_cost = (
    input_tokens * PRICE_INPUT_PER_M + output_tokens * PRICE_OUTPUT_PER_M
) / 1_000_000

print("Seconds per review:", round(api_seconds, 2))
print("Input tokens:", input_tokens, "| output tokens:", output_tokens)

Seconds per review: 1.18
Input tokens: 1853 | output tokens: 203


## Schritt 3: Genauigkeit, Geschwindigkeit und Kosten vergleichen

In [7]:
# Accuracy against the silver labels, time per review and API cost
comparison = pd.DataFrame(
    {
        LOCAL_MODEL: [
            accuracy_score(test["silver"], test["local"]),
            local_seconds,
            0.0,
        ],
        MODEL: [
            accuracy_score(test["silver"], test["api"]),
            api_seconds,
            api_cost,
        ],
    },
    index=[
        "accuracy (silver labels)",
        "seconds per review",
        f"API cost in USD ({len(test)} reviews)",
    ],
)
comparison.round(4)

,qwen3:1.7b,gpt-6-luna
accuracy (silver labels),0.8000,0.8000
seconds per review,5.3224,1.1804
API cost in USD (10 reviews),0.0000,0.0003


In [8]:
# Reviews where at least one model disagrees with the silver label
pd.set_option("display.max_colwidth", 300)
errors = test[
    (test["local"] != test["silver"]) | (test["api"] != test["silver"])
]
errors[["id", "silver", "local", "api", "text"]]

,id,silver,local,api,text
2,R152,mixed,negative,negative,"Wow, the customer service here has fallen apart. I had 4 employee interactions and 3 were terrible. The first was with a person at customer service in the warehouse area. I asked him where their very common <PERSON> shelving was. He had never heard of it and pointed at a broken computer and sai..."
8,R347,positive,mixed,mixed,"Clean efficient <PERSON>, newly renovated but in a rough part of downtown near the St Joseph Shrine. \n\nEasy in and out, can order at the automated kiosk and have dedicated staff to keep the dining room clean. \n\nAlso there is a 30 minute limit to how long you can be here, and no outside food ..."


## Den Vergleich interpretieren

**Genauigkeit (Accuracy):**

Betrachte die gemessenen Genauigkeitswerte und untersuche die unterschiedlichen Klassifizierungen unterhalb der Tabelle.

Welches Modell macht in dieser Stichprobe mehr Fehler?

Gehe nicht automatisch davon aus, dass das grössere Modell besser ist.

**Fairer Vergleich?**

Die Silver Labels wurden vom Modell gpt-6-luna selbst erstellt.

Ein Modell stimmt möglicherweise leichter mit seinen eigenen Referenzlabels überein. Dadurch kann es beim Vergleich einen Vorteil haben.

Für einen faireren Vergleich benötigen wir Gold Labels, die unabhängig von Menschen erstellt und überprüft wurden.

**10 Rezensionen sind eine kleine Stichprobe:**

Eine einzige zusätzliche richtige oder falsche Klassifizierung verändert die Accuracy bereits um 10 Prozentpunkte.

Für eine zuverlässige Entscheidung müsste man deshalb deutlich mehr Rezensionen testen.

**Kosten (Cost):**

Ein lokales Modell verursacht keine direkten Kosten pro Token.

Das bedeutet jedoch nicht, dass es kostenlos betrieben werden kann.

Es benötigt Hardware, Strom und jemanden, der das Modell installiert, betreibt und wartet.

**Datenschutz (Privacy):**

Bei einem vollständig lokal betriebenen Modell verlassen die Rezensionstexte den eigenen Computer nicht.

Das kann ein entscheidender Vorteil sein, insbesondere wenn sensible oder vertrauliche Daten verarbeitet werden.

## Überprüfe dein Verständnis

Beantworte die folgenden Fragen kurz anhand der oben erzeugten Ergebnisse.

Versuche zuerst, die Fragen selbst zu beantworten, und öffne anschliessend die Hinweise.

1. Was verändert sich, wenn der Client auf localhost statt auf die externe API zugreift? Wo werden die Rezensionstexte jeweils verarbeitet?

2. Betrachte deine Vergleichstabelle: Welches Modell ist genauer und welches ist schneller? Kannst du aus diesen Ergebnissen schliessen, dass eines der Modelle grundsätzlich immer besser ist?

3. Warum können Silver Labels, die vom API-Modell selbst erstellt wurden, dieses Modell beim Vergleich begünstigen? Wie könnte man unabhängigere Referenzlabels erstellen?

4. Warum beschreiben die gemessenen Ausführungszeiten nur die Leistung dieses Computers zum Zeitpunkt des Tests? Warum bedeuten API-Kosten von null nicht, dass der Betrieb eines lokalen Modells kostenlos ist?

5. Nenne eine Anforderung, die für eine lokale Verarbeitung spricht, und eine Anforderung, die für die Nutzung einer API spricht. Welche einfachere Vergleichsmethode (Baseline) könntest du testen, bevor du dich für eines der beiden Modelle entscheidest?

**Deine Antworten:**

1. Die Basis-URL (base_url) und das verwendete Modell ändern sich. Ollama stellt das lokale Modell auf dem eigenen Computer bereit. Bei der externen API werden die übermittelten Texte dagegen auf den Systemen des API-Anbieters verarbeitet.
2. Verwende die tatsächlich gemessenen Werte, auch wenn sie von deinen Erwartungen abweichen. Zehn Rezensionen und die Messungen auf einem einzigen Computer sind keine ausreichende Grundlage für allgemeingültige Aussagen über die Qualität oder Geschwindigkeit der Modelle.
3. Das API-Modell kann seine eigenen Referenzlabels möglicherweise leichter reproduzieren. Unabhängig erstellte und von Menschen überprüfte Gold Labels bieten eine fairere Vergleichsgrundlage.
4. Die Ausführungszeit hängt unter anderem von der Hardware, der Netzwerkverbindung und der aktuellen Auslastung zum Zeitpunkt des Tests ab. Der lokale Betrieb verursacht weiterhin Kosten für Hardware, Strom und Wartung.
5. Beispielsweise spricht die Anforderung, dass vertrauliche Texte den eigenen Computer nicht verlassen dürfen, für eine lokale Verarbeitung. Begrenzte Hardware oder eine geringere gemessene Antwortzeit können für eine API sprechen. Als einfache Baseline könnte man zunächst TF-IDF in Kombination mit einem klassischen Klassifikationsmodell testen.

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hinweise zur Selbstkontrolle anzeigen</summary>

1. Die Basis-URL (base_url) und das verwendete Modell ändern sich. Ollama stellt das lokale Modell auf dem eigenen Computer bereit. Bei der externen API werden die übermittelten Texte dagegen auf den Systemen des API-Anbieters verarbeitet.

2. Verwende die tatsächlich gemessenen Werte, auch wenn sie von deinen Erwartungen abweichen. Zehn Rezensionen und die Messungen auf einem einzigen Computer sind keine ausreichende Grundlage für allgemeingültige Aussagen über die Qualität oder Geschwindigkeit der Modelle.

3. Das API-Modell kann seine eigenen Referenzlabels möglicherweise leichter reproduzieren. Unabhängig erstellte und von Menschen überprüfte Gold Labels bieten eine fairere Vergleichsgrundlage.

4. Die Ausführungszeit hängt unter anderem von der Hardware, der Netzwerkverbindung und der aktuellen Auslastung zum Zeitpunkt des Tests ab. Der lokale Betrieb verursacht weiterhin Kosten für Hardware, Strom und Wartung.

5. Beispielsweise spricht die Anforderung, dass vertrauliche Texte den eigenen Computer nicht verlassen dürfen, für eine lokale Verarbeitung. Begrenzte Hardware oder eine geringere gemessene Antwortzeit können für eine API sprechen. Als einfache Baseline könnte man zunächst TF-IDF in Kombination mit einem klassischen Klassifikationsmodell testen.

</details>

## Optional: Eine grössere Stichprobe testen

Wenn du genügend Zeit hast, kannst du in Schritt 2 die Anzahl der ausgewählten Rezensionen erhöhen.

Ändere dazu:

`sample(n=5, ...)`

zu:

`sample(n=15, ...)`

Damit werden pro Unternehmenskette 15 statt 5 Rezensionen ausgewählt, sofern genügend Rezensionen mit gültigen Silver Labels verfügbar sind.

Führe anschliessend den Vergleich erneut durch.

Frage: Werden die bisherigen Schlussfolgerungen durch die grössere Stichprobe weiterhin bestätigt?

Beachte, dass dieser Test länger dauert und zusätzliche API-Anfragen verursacht.

Weitere Erweiterungen sind für den grundlegenden Modellvergleich nicht erforderlich.

### Jupyter-Notebook – Informationen in der Fusszeile (bitte immer am Ende jedes Notebooks angeben)

In [9]:
# Show operating system, date/time and Python version
print("-----------------------------------")
print(os.name.upper())
print(platform.system(), "|", platform.release())
print("Datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python Version:", python_version())
print("-----------------------------------")


-----------------------------------
POSIX
Linux | 6.8.0-1070-azure
Datetime: 2026-11-16 09:58:47
Python Version: 3.11.18
-----------------------------------
